# Re-run the chemCPA conditional-expert study

A thin wrapper around the scripts the study ran. It submits nothing by itself; it lists the
exact job sequence and then reads the small result tables that live in Git. The research is in
`reports/chemcpa_moe_v2/RESULTS.md`, not here.

**Order matters.** Step 8 is the only step that opens the held-out test drugs, and it runs after
the arms are frozen. Running it earlier would put test scores inside the selection procedure.

> **2026-11-13:** the arm numbers read under *The arms* below come from the first pass, whose evaluator gave 43 of the 48
> scored cell lines another line's covariate one-hot. They are kept as the record and are **withdrawn**. The
> corrected re-run is read in the section *Corrected re-run* at the end, and written up in
> `reports/chemcpa_moe_v2/RESULTS_FIXED.md`.


In [ ]:
ROOT = "/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver_chemcpa_moe_v2"
BRANCH = "research/chemcpa-conditional-moe-v2"
print(f"cd {ROOT}  # git checkout {BRANCH}")

## The pipeline, as it was actually submitted

Each line is one Slurm job. Times are the measured ones, on one A30 or A40.

```bash
cd $ROOT        # the server worktree; env.sh sets every path below

# 0. the initialisation fix, on the unchanged v1 table (goal section 0)
sbatch experiments/initfix_v2/checks.sbatch                       #  ~2 min
sbatch experiments/initfix_v2/rerun.sbatch                        #  ~2 min, 6 runs

# 1. public chemCPA, set up and run once on the authors' own example (goal section 2)
sbatch experiments/chemcpa_moe_v2/chemcpa_example.sbatch
sbatch experiments/chemcpa_moe_v2/example_report.sbatch

# 2. the v2 data: one pass over the shards, then the table and the cells (goal section 3)
sbatch experiments/chemcpa_moe_v2/build_v2.sbatch                 #  1,475 s, CPU
sbatch experiments/chemcpa_moe_v2/prepare_v2.sbatch

# 3. baselines and the v1 architectures, re-scored on v2
sbatch experiments/chemcpa_moe_v2/baselines_v2.sbatch

# 4. the five pre-run checks -- a failure here is a failure of the comparison
sbatch experiments/chemcpa_moe_v2/checks_cpa.sbatch

# 5. the learning-rate ladder at seed 42, then the selected scale at seeds 43/44
for A in P0 P1 P2; do sbatch experiments/chemcpa_moe_v2/sweep_cpa.sbatch  $A; done
for A in P0 P1 P2; do sbatch experiments/chemcpa_moe_v2/seeds_cpa.sbatch  $A 3.0; done
sbatch experiments/chemcpa_moe_v2/aggregate_cpa.sbatch            # selection + the P3 gate

# 6. why the arms score where they do, on the frozen checkpoints
sbatch experiments/chemcpa_moe_v2/diagnose_cpa.sbatch

# 7. the same setting at three times the budget, for all three arms equally
for A in P0 P1 P2; do sbatch experiments/chemcpa_moe_v2/budget_cpa.sbatch $A 3.0 30; done
sbatch experiments/chemcpa_moe_v2/diagnose_cpa.sbatch "$PWD/assets/v2/runs_e30" \
       reports/chemcpa_moe_v2/diagnostics_e30.json

# 8. the single test pass -- refuses to run with uncommitted code under src/experiments/configs
sbatch experiments/chemcpa_moe_v2/final_eval_cpa.sbatch
```

Total measured training cost: **20.1 GPU-hours**.


## The initialisation fix (goal section 0)

In [ ]:
import json
import pandas as pd

R0 = "reports/initfix_v2"
runs = pd.read_csv(f"{R0}/runs.tsv", sep="\t")
display(runs)
s = json.load(open(f"{R0}/validation_summary.json"))
print("A and C start from identical shared weights:", s["shared_init_matches_across_kinds"])
for k in ["A", "C"]:
    print(k, "seed mean of scores", s[k]["seed_mean_of_scores"],
          "| score of mean prediction", s[k]["score_of_mean_prediction"])
print("C over A:", s["C_over_A_percent"])

The two aggregates are different statistics and are kept apart everywhere:
`seed_mean_of_scores` scores each seed's model and averages the scores, which is the
pre-registered primary; `score_of_mean_prediction` averages three seeds' predictions and scores
once, which is a three-model ensemble and was never pre-registered.

## What the official chemCPA example showed

In [ ]:
ex = json.load(open("reports/chemcpa_moe_v2/chemcpa_example.json"))
print(json.dumps(ex["io"], indent=1))
print(json.dumps(ex["dataset"], indent=1))
print(ex["objective"])

## The v2 data

In [ ]:
t = json.load(open("manifests/chemcpa_moe_v2/transform.json"))
print(t["truth"]); print("v1 was:", t["v1_truth"])
print({k: t[k] for k in ["n_valid_conditions", "n_valid_train", "n_valid_val", "n_valid_test",
                         "truth_rms_against_zero"]})
print(json.load(open("manifests/chemcpa_moe_v2/anndata.json")))

## The arms

In [ ]:
R = "reports/chemcpa_moe_v2"
trials = pd.read_csv(f"{R}/trials.tsv", sep="\t")          # the 30-epoch runs
display(trials[["arm", "seed", "val_macro_delta_rmse", "best_epoch", "params_total",
                "params_residual", "train_seconds", "base_init_sha256"]])

# the fairness precondition: one initialisation per seed, shared by every arm
print(trials.groupby("seed").base_init_sha256.nunique())   # 1 per seed
print(trials.base_init_sha256.nunique())                   # 3 across seeds

sel = json.load(open(f"{R}/selection.json"))
print(json.dumps(sel["gate"], indent=1))                   # train_P3 is False


In [ ]:
# where each arm's score comes from: observed control (the primary) vs the model's own control
import statistics as st
d = json.load(open(f"{R}/diagnostics_e30.json"))
rows = []
for arm, v in d["control_modes"].items():
    per = v["per_seed"].values()
    rows.append(dict(arm=arm,
                     observed=round(st.mean(r["observed"] for r in per), 4),
                     model_control=round(st.mean(r["model"] for r in per), 4),
                     reconstruction_pct=round(st.mean(r["reconstruction_share_pct"] for r in per), 1),
                     rms_predicted_response=round(st.mean(r["rms_predicted_response_model"] for r in per), 4),
                     rms_truth=round(list(per)[0]["rms_truth"], 4)))
display(pd.DataFrame(rows))
# no-change scores 0.1028 here: under its own control every arm predicts almost no drug effect


In [ ]:
# the single test pass, with the baselines scored on the same conditions
test = json.load(open(f"{R}/test_results.json"))
display(pd.DataFrame({a: {k: v for k, v in r.items() if not isinstance(v, dict)}
                      for a, r in test["results"].items()}).T)
print(json.dumps(test["reference"], indent=1))
print(json.dumps(test["bootstrap"], indent=1))
for a, r in test["results"].items():
    print(a, "plate-centred:", r["plate_diagnostic"])

mlp = json.load(open(f"{R}/mlp_test/test_results.json"))["summary"]
display(pd.DataFrame({k: {m: v[m] for m in ["macro_delta_rmse", "drug_centred_r",
                                            "top20_direction", "snu423_delta_rmse"]}
                      for k, v in mlp.items()}).T)


Read those two tables together. The arms order themselves P2 < P1 < P0 on both validation and
test, and their drug-centred correlation is two to seven times the baselines'. They are also all
about three times worse than predicting a per-line constant, and the diagnostic above says why:
roughly two thirds of the score is absolute-profile error, and once that is removed the arms sit
on the no-change baseline. That is why `main_effect` is negative and `routing_effect` is
`not_tested` — the gate refused to promote P2, so P3 was never trained.

## What this notebook is not

It does not select anything, fit a corrector on test truth, or produce a figure that is not
already in the reports. Changing a setting here starts a new study, which needs its own
pre-registration and its own held-out split.

## Corrected re-run

Same arms plus P2b (basal-latent router), corrected evaluator, learning rate re-selected on validation, 30 epochs ×
seeds 42/43/44. The test pass is a bug-fix re-evaluation with checkpoint hashes committed beforehand.


In [ ]:
F = "reports/chemcpa_moe_v2/fixed"
sel = json.load(open(f"{F}/e30/selection.json"))
print(pd.DataFrame({a: v["per_seed_score"] | {"seed_mean": v["seed_mean_of_scores"]}
                    for a, v in sel["aggregates"].items()}).T)        # validation
print("baselines (validation):", sel["baselines"])
print("gate:", json.dumps(sel["gate"], indent=1))


In [ ]:
test = json.load(open(f"{F}/test_results.json"))
pre = json.load(open("manifests/chemcpa_moe_v2/freeze_fixed_pretest.json"))["checkpoint_sha256"]
print("checkpoints scored == hashes committed before the pass:", test["checkpoint_sha256"] == pre)
rows = {a: dict(seed_mean=v["seed_mean_of_scores"], **{k: v["secondaries_seed_mean"][k] for k in
        ["drug_centred_r", "top20_direction", "snu423_delta_rmse"]}) for a, v in test["results"].items()}
rows |= {b: dict(seed_mean=s["macro_delta_rmse"], drug_centred_r=s["drug_centred_r"],
        top20_direction=s["top20_direction"], snu423_delta_rmse=s["snu423_delta_rmse"])
        for b, s in test["baselines_same_rows"].items()}
print(pd.DataFrame(rows).T)                                       # test, same rows for everyone
print(pd.DataFrame(test["bootstrap"]["primary_seed_mean_of_scores"]).T)
